# Ten-year PMC drug–gene extraction

Publication window: **2016-11-03 through 2026-11-03**, inclusive. Discover the full PMC cohort, use all parsed abstract/body sections, identify normalized drug and gene candidates with BioBERT/VICC, and classify each section pair with Qwen3-14B on Apple Silicon Metal.

This is a corpus run, not a labeled evaluation. Outputs are model predictions for review; precision and recall cannot be calculated without a truth set. Full-text and abstract-only coverage are reported separately. Only successfully normalized entities enter classification; exclusions and failures remain auditable.

Install into this notebook's native arm64 Python environment using `pip install -e '.[pmc,qwen]'` from the repository root. Optional notebook dependencies: `nbformat`, `ipykernel`. Set `MC_EMAIL` or `NCBI_EMAIL` for NCBI contact information; `NCBI_API_KEY` is optional. The pipeline never logs these values. No PubTator file or gene panel is needed.

**The complete cohort may require a very long run on one Mac.** The resumable CLI is preferable for unattended processing. Do not run two writers against the same output directory.

In [ ]:
from pathlib import Path
import json
import platform

from dgilit.pmc_pipeline import PMCPipeline, PipelineConfig, DEFAULT_OUTPUT

assert platform.system() == "Darwin" and platform.machine() == "arm64", "Local inference requires Apple Silicon macOS"
config = PipelineConfig(start="2016-11-03", end="2026-11-03")
pipeline = PMCPipeline(output=DEFAULT_OUTPUT, config=config)
print(f"Output: {pipeline.output}")
pipeline.status()

## 1. Discover the complete cohort

Discovery uses PMC publication-date queries, recursively splitting large ranges, validating page counts, and persisting PMCIDs. Papers without PMIDs are retained. This step can take time for millions of records; rerunning resumes completed partitions. A changing search result produces an explicit incomplete-discovery error rather than silently accepting missing IDs.

In [ ]:
pipeline.discover()

## 2. Run the pilot

Deterministically sample 100 articles, balanced across publication-year strata. Fetch and prepare those articles, and initially measure at most 100 Qwen context tasks. Read the resulting throughput/storage estimates and coverage counts. All pilot work is reused by the full run.

NER uses overlapping tokenizer windows, and oversized Qwen contexts are split with overlap. Every normalized drug×gene pair in a section is evaluated against each of that section's Qwen contexts; cross-section inference is not performed. Downloaded text is cached within 10 GiB, with completed articles evicted first. Processing checkpoints if free disk space falls below 10 GiB.

In [ ]:
pipeline.run(pilot_only=True)

In [ ]:
pipeline.status()["pilot"]

## 3. Continue the complete corpus

The following cell runs until the current pass finishes or is interrupted/resource-blocked. It has **no default article or inference cap**. Interrupting and rerunning reuses completed work. Prefer the CLI for long unattended runs:

```bash
python -m dgilit.pmc_cli run --detach
python -m dgilit.pmc_cli status
```

`status.json` is an atomic progress snapshot; the `status` command reads current SQLite state. Explicit diagnostic controls `--pilot-only` and `--max-calls N` are available. `--max-calls` applies after the initial pilot.

In [ ]:
pipeline.run()

## 4. Inspect coverage and retry failures

A transient exception is never a negative interaction. At article level any supported context yields a positive pair; negatives require all required contexts to succeed negatively. Failed tasks, normalization failures, unavailable text, and no-candidate sections remain distinguishable. Positives require an evidence quote verifiable in their source context.

Run the following commands when retrying unresolved failures:

```bash
python -m dgilit.pmc_cli retry-failed
python -m dgilit.pmc_cli run
```

Retries keep successful task results. Changing model, prompt, package versions, or run configuration requires a new output directory.

In [ ]:
pipeline.status()

## 5. Export Parquet datasets

Exports stream from SQLite into compressed Parquet shards. Each snapshot contains article inventory, discovery partitions, section provenance, entities, all context classifications, article pairs (including incomplete pairs), positive interactions, and failures. Source-text hashes and task IDs support traceability after text-cache eviction. `result`/`data` fields retain full structured records as JSON; explicit columns support routine filtering.

`positive_interactions` includes positive pairs even if other contexts remain unresolved; inspect `unresolved_contexts` and `article_status`. `supporting_task_ids` links to evidence in `classifications`; `pair_status` distinguishes positive, negative, and unresolved pairs. Full text may need to be fetched again to reproduce an evicted context. No Excel export is attempted for this corpus size.

In [ ]:
manifest = pipeline.export()
print(json.dumps(manifest["datasets"], indent=2))

In [ ]:
import polars as pl
pointer = json.loads((pipeline.output / "exports/latest.json").read_text())
snapshot = Path(pointer["manifest"]).parent
positive_interactions = pl.scan_parquet(str(snapshot / "positive_interactions/*.parquet"))
positive_interactions.head(20).collect()